# TFP-BD 80/10/10 training: YOLOv8n + YOLO11n

**Nijer data preparation** (original XML -> YOLO) + **80/10/10 split** (80% train) e YOLOv8n, YOLO11n train kora hoy - **70/20/10 er HUBOHU same training setting**
(epoch, batch, augmentation, learning rate sob ager weight file er `train_args` theke pora hoy). Parthokko shudhu training data-r poriman.

- Test = Sharmin er same 2,368 test chobi (70/20/10 er sathe sorasori tulona)
- Shesh e Ultralytics test mAP (quick check); puro encroachment/congestion hobe Evaluation notebook e

**Input:** Bangladeshi Traffic Flow Dataset (nijer), Data Preparation (shudhu common test list) + ager weight: Sharmin_yolov8_7030, yolov11 (Models). **GPU T4 x2, Internet ON.**
Cell 1-2 chalie check, tarpor **Save Version -> Save & Run All**. Somoy: ~3.5-4 ghonta.

In [ ]:
# CELL 1: install + input
!pip install -q ultralytics
import os, glob, shutil, json, time, random
import torch, yaml
import pandas as pd
import ultralytics
from ultralytics import YOLO, RTDETR
from ultralytics.cfg import DEFAULT_CFG_DICT
print("ultralytics", ultralytics.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NAI!")
WK = "/kaggle/working"

SH = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "tfp_dataset" in dirs:
        c = os.path.join(root, "tfp_dataset")
        if all(os.path.isdir(os.path.join(c, s, "images")) for s in ("train", "val", "test")):
            SH = c
            break
    dirs[:] = [d for d in dirs if d not in ("images", "labels", "Bangladeshi Traffic Flow Dataset")]
assert SH, "tfp_dataset pawa jayni - Add Input e Sharmin er 'Data Preparation' notebook jog koro (shudhu test-er file naam er jonno)"
ORIG = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "Bangladeshi Traffic Flow Dataset" in dirs:
        c = os.path.join(root, "Bangladeshi Traffic Flow Dataset")
        if os.path.isdir(os.path.join(c, "Annotated Images")):
            ORIG = c
            break
    dirs[:] = [d for d in dirs if d not in ("images", "labels", "tfp_dataset")]
assert ORIG, "Original 'Bangladeshi Traffic Flow Dataset' pawa jayni - Add Input e jog koro"
print("Original dataset (nijer):", ORIG)
print("Common test list:", SH + "/test")

OLD = {}
for root, dirs, files in os.walk("/kaggle/input"):
    dirs[:] = [d for d in dirs if d not in ("images", "labels", "tfp_dataset", "Bangladeshi Traffic Flow Dataset")]
    for f in files:
        p = os.path.join(root, f)
        low = p.lower()
        if f == "sharmin_yolov8n_7030_best.pt":
            OLD["YOLOv8n"] = p
        elif f == "best.pt" and "yolov11" in low:
            OLD["YOLO11n"] = p
        elif f == "best.pt" and "rtdetr" in low:
            OLD["RT-DETR-L"] = p
JOBS = [('YOLOv8n', 'YOLO', 'yolov8n.pt'), ('YOLO11n', 'YOLO', 'yolo11n.pt')]
for tag, cls_name, base in JOBS:
    print(f"{'OK ' if tag in OLD else 'MISSING'}  {tag:10s} old weight: {OLD.get(tag, '')}")
assert all(t in OLD for t, _, _ in JOBS), "ager weight pawa jayni - Add Input e model ta jog koro"

In [ ]:
# CELL 2: 80/10/10 split banano (symlink, disk e copy hoy na)

# ---- NIJER data preparation (original XML theke YOLO label) + 80/10/10 split
# test = team er common test set (70/20/10 er same 2,368 chobi, shudhu file-er NAAM Sharmin er folder theke pora hoy)
# train/val = baki 21,310 chobi theke NIJER random split (seed 42): val 2,368 (10%), train 18,942 (80%)
import random, glob
import xml.etree.ElementTree as ET
EXTS = (".jpg", ".jpeg", ".png", ".bmp")
SPLIT_ROOT = "/kaggle/working/tfp_8010"
CLS9 = ["Car", "Bus", "Truck", "Motorcycle", "Bicycle", "Rickshaw", "Mini-Truck", "CNG", "Pedestrian"]
NAME_MAP = {"car": "Car", "bus": "Bus", "truck": "Truck", "bike": "Motorcycle", "cycle": "Bicycle",
            "rickshaw": "Rickshaw", "mini-truck": "Mini-Truck", "cng": "CNG", "people": "Pedestrian"}

def voc_to_yolo_lines(xml_path):
    root = ET.parse(xml_path).getroot()
    size = root.find("size")
    W, H = float(size.find("width").text), float(size.find("height").text)
    lines = []
    for obj in root.findall("object"):
        key = str(obj.find("name").text).strip().lower()
        if key not in NAME_MAP:
            continue
        b = obj.find("bndbox")
        x0, y0, x1, y1 = [float(b.find(t).text) for t in ("xmin", "ymin", "xmax", "ymax")]
        lines.append(f"{CLS9.index(NAME_MAP[key])} {((x0 + x1) / 2) / W:.6f} {((y0 + y1) / 2) / H:.6f} {(x1 - x0) / W:.6f} {(y1 - y0) / H:.6f}")
    return lines

def build_split_8010(SH, out_root=SPLIT_ROOT, make_links=True):
    # 1) original dataset theke image + XML jora
    pairs = {}
    for xf in glob.glob(os.path.join(ORIG, "**", "*.xml"), recursive=True):
        st = os.path.splitext(os.path.basename(xf))[0]
        for ext in (".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"):
            ip = os.path.join(os.path.dirname(xf), st + ext)
            if os.path.exists(ip):
                pairs[st] = (ip, xf)
                break
    # 2) common test set (shudhu naam)
    test = sorted(os.path.splitext(f)[0] for f in os.listdir(f"{SH}/test/images") if f.lower().endswith(EXTS))
    assert all(t in pairs for t in test), "test chobi original dataset e pawa jayni"
    pool = sorted(set(pairs) - set(test))
    order = pool[:]
    random.Random(42).shuffle(order)
    n_val = len(test)
    S = {"train": sorted(order[n_val:]), "val": sorted(order[:n_val]), "test": test}
    if make_links:
        for s, stems in S.items():
            os.makedirs(f"{out_root}/{s}/images", exist_ok=True)
            os.makedirs(f"{out_root}/{s}/labels", exist_ok=True)
            for st in stems:
                ip, xf = pairs[st]
                dst = f"{out_root}/{s}/images/{os.path.basename(ip)}"
                if not os.path.lexists(dst):
                    os.symlink(ip, dst)
                with open(f"{out_root}/{s}/labels/{st}.txt", "w") as fh:
                    fh.write("\n".join(voc_to_yolo_lines(xf)) + "\n")
    tot = sum(len(v) for v in S.values())
    print(f"Original dataset: {len(pairs)} image+XML jora")
    print("Nijer 80/10/10 split:", {k: f"{len(v)} ({len(v) / tot * 100:.1f}%)" for k, v in S.items()})
    # 3) sanity: amar conversion ar Sharmin er label ki eki? (200 ta test file)
    if make_links:
        same, chk = 0, test[:200]
        for st in chk:
            mine = open(f"{out_root}/test/labels/{st}.txt").read().split()
            hers = open(f"{SH}/test/labels/{st}.txt").read().split()
            same += mine == hers
        print(f"  Label conversion check: {same}/{len(chk)} file Sharmin er conversion er sathe hubohu mile")
    return S, pairs

S, SRC = build_split_8010(SH)
CLASSES = ["Car", "Bus", "Truck", "Motorcycle", "Bicycle", "Rickshaw", "Mini-Truck", "CNG", "Pedestrian"]
YAML = f"{WK}/data_8010.yaml"
yaml.safe_dump({"train": f"{SPLIT_ROOT}/train/images", "val": f"{SPLIT_ROOT}/val/images",
                "test": f"{SPLIT_ROOT}/test/images", "nc": 9, "names": CLASSES}, open(YAML, "w"))
json.dump(S, open(f"{WK}/split_8010_lists.json", "w"))
print("data yaml:", YAML)

In [ ]:
# CELL 3: ager weight theke training setting pora (same hyper-parameter)
DROP = {"task", "mode", "model", "data", "project", "name", "exist_ok", "resume", "device", "save_dir",
        "source", "workers", "save_period", "plots", "val", "split", "cache"}
KEY_SHOW = ["epochs", "batch", "imgsz", "optimizer", "lr0", "lrf", "mosaic", "mixup", "copy_paste", "close_mosaic", "seed"]
SETTINGS = {}
for tag, cls_name, base in JOBS:
    cls = RTDETR if cls_name == "RTDETR" else YOLO
    m = cls(OLD[tag])
    ta = dict((m.ckpt or {}).get("train_args", {}))
    del m
    b = os.path.basename(str(ta.get("model") or base))
    if b not in (base,):
        print(f"  [{tag}] ager base model '{ta.get('model')}' -> standard '{base}' use kora hobe")
        b = base
    kw = {k: v for k, v in ta.items() if k not in DROP and k in DEFAULT_CFG_DICT}
    SETTINGS[tag] = (cls_name, b, kw)
    print(f"[{tag}] base {b} | ager data: {ta.get('data')}")
    print("   ", {k: kw.get(k) for k in KEY_SHOW})
json.dump({t: {"base": v[1], "args": {k: str(x) for k, x in v[2].items()}} for t, v in SETTINGS.items()},
          open(f"{WK}/train_settings_8010.json", "w"), indent=1)

In [ ]:
# CELL 4: YOLOv8n train (80/10/10) ~1.5-2 h
TAG = "YOLOv8n"
cls_name, base, kw = SETTINGS[TAG]
cls = RTDETR if cls_name == "RTDETR" else YOLO
safe = TAG.replace("-", "_")
DEVICE = 0
if isinstance(DEVICE, list):
    assert torch.cuda.device_count() >= len(DEVICE), "2 ta GPU pawa jayni - Session options e Accelerator: GPU T4 x2 select koro"
    print(f"Duita GPU ekshathe (DDP): {DEVICE} | total batch {kw.get('batch')} same thake, protita GPU te {int(kw.get('batch', 16)) // len(DEVICE)}")
t0 = time.time()
model = cls(base)
model.train(data=YAML, project=f"{WK}/runs", name=f"{safe}_8010", exist_ok=True, device=DEVICE, workers=4,
            save_period=-1, plots=True, **kw)
best = f"{WK}/runs/{safe}_8010/weights/best.pt"
shutil.copy(best, f"{WK}/{safe}_8010_best.pt")
print(f"[{TAG}] training done in {(time.time() - t0) / 3600:.2f} h -> {safe}_8010_best.pt")
del model
torch.cuda.empty_cache()

In [ ]:
# CELL 5: YOLO11n train (80/10/10) ~1.5-2 h
TAG = "YOLO11n"
cls_name, base, kw = SETTINGS[TAG]
cls = RTDETR if cls_name == "RTDETR" else YOLO
safe = TAG.replace("-", "_")
DEVICE = 0
if isinstance(DEVICE, list):
    assert torch.cuda.device_count() >= len(DEVICE), "2 ta GPU pawa jayni - Session options e Accelerator: GPU T4 x2 select koro"
    print(f"Duita GPU ekshathe (DDP): {DEVICE} | total batch {kw.get('batch')} same thake, protita GPU te {int(kw.get('batch', 16)) // len(DEVICE)}")
t0 = time.time()
model = cls(base)
model.train(data=YAML, project=f"{WK}/runs", name=f"{safe}_8010", exist_ok=True, device=DEVICE, workers=4,
            save_period=-1, plots=True, **kw)
best = f"{WK}/runs/{safe}_8010/weights/best.pt"
shutil.copy(best, f"{WK}/{safe}_8010_best.pt")
print(f"[{TAG}] training done in {(time.time() - t0) / 3600:.2f} h -> {safe}_8010_best.pt")
del model
torch.cuda.empty_cache()

In [ ]:
# CELL LAST: Ultralytics quick test mAP (same Sharmin test set) - shudhu check; final number Evaluation notebook e
rows = []
for tag, cls_name, base in JOBS:
    safe = tag.replace("-", "_")
    cls = RTDETR if cls_name == "RTDETR" else YOLO
    r = cls(f"{WK}/{safe}_8010_best.pt").val(data=YAML, split="test", imgsz=640, batch=16, plots=False,
                                              project=f"{WK}/runs", name=f"{safe}_8010_test", exist_ok=True)
    rows.append({"model": tag, "split": "80/10/10", "ultra_test_mAP50": r.box.map50, "ultra_test_mAP50-95": r.box.map,
                 "precision": r.box.mp, "recall": r.box.mr})
ref = {"YOLOv8n": (0.7441, 0.5124), "YOLO11n": (0.7299, 0.4974), "RT-DETR-L": (0.7339, 0.5076)}
for tag, _, _ in JOBS:
    rows.append({"model": tag, "split": "70/20/10 (Part 1)", "ultra_test_mAP50": ref[tag][0], "ultra_test_mAP50-95": ref[tag][1]})
df = pd.DataFrame(rows).sort_values(["model", "split"])
df.to_csv(f"{WK}/quick_test_8010.csv", index=False)
print(df.round(4).to_string(index=False))
for f in glob.glob(f"{WK}/runs/*_8010/results.csv"):
    shutil.copy(f, f"{WK}/" + f.split("/")[-2] + "_results.csv")
print("\nOutput e ache:", sorted(x for x in os.listdir(WK) if x.endswith((".pt", ".csv", ".json"))))